In [1]:
pip install transformers torch


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install easyocr opencv-python ultralytics


Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install --upgrade optree


In [ ]:
import cv2
import easyocr
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from ultralytics import YOLO
import os

ui_model = YOLO("yolov8n.pt")  # Pretrained YOLOv8 model

# Load DeepSeek Model
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Load OCR Model
reader = easyocr.Reader(["en"])

# preprocess image
def preprocess_image(image):
    """Apply preprocessing steps to enhance UI detection and OCR accuracy."""
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)

    processed = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 11, 2)

    return processed

def process_ui_image(image_path):

    if not os.path.exists(image_path):
        print(f"❌ Error: Image not found at {image_path}")
        return []

    # Load and preprocess image
    image = cv2.imread(image_path)
    if image is None:
        print("❌ Error: Unable to load image!")
        return []

    image = cv2.resize(image, (1280, 720))
    processed_image = preprocess_image(image)

    results = ui_model(image)

    components = []
    for result in results:
        for box in result.boxes.xyxy: 
            x1, y1, x2, y2 = map(int, box)
            cropped = image[y1:y2, x1:x2]
            
            cropped_processed = preprocess_image(cropped)

            text_results = reader.readtext(cropped_processed)
            text = text_results[0][-2] if text_results else "Unknown"

            components.append({
                "type": "button" if "button" in text.lower() else "text_field",
                "text": text,
                "position": {"x": x1, "y": y1},
                "size": {"width": x2-x1, "height": y2-y1}
            })

    print("\n✅ UI Components Extracted:", json.dumps(components, indent=2))
    return components

# Function to generate JSON using DeepSeek
def generate_ui_json(components):
    
    prompt = f"""
    Convert these UI components into JSON format:
    
    Input:
    {json.dumps(components, indent=2)}

    Output:
    {{
      "UI": {{
        "components": [
          {{
            "type": "text_field",
            "text": "Enter Username",
            "position": {{"x": 50, "y": 100}},
            "size": {{"width": 300, "height": 40}}
          }}
        ]
      }}
    }}
    """

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output = model.generate(input_ids, max_length=500, do_sample=True, temperature=0.5)

    json_output = tokenizer.decode(output[0], skip_special_tokens=True)
    print("\n🟡 Raw DeepSeek Output:", json_output)

    # Ensure JSON is parsed correctly
    try:
        json_data = json.loads(json_output)
        return json_data
    except json.JSONDecodeError:
        return {"error": "Invalid JSON output"}


image_path = "D:\\ui image 2.jpg"

ui_components = process_ui_image(image_path)
if ui_components:
    ui_json = generate_ui_json(ui_components)

    # Save JSON output
    with open("ui_output.json", "w") as f:
        json.dump(ui_json, f, indent=2)

    print("\n✅ JSON file saved as ui_output.json")
else:
    print("❌ No UI elements detected in the image.")



0: 384x640 1 surfboard, 93.4ms
Speed: 12.5ms preprocess, 93.4ms inference, 166.2ms postprocess per image at shape (1, 3, 384, 640)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



✅ UI Components Extracted: [
  {
    "type": "text_field",
    "text": "5m323e",
    "position": {
      "x": 469,
      "y": 221
    },
    "size": {
      "width": 325,
      "height": 104
    }
  }
]

🟡 Raw DeepSeek Output: 
    Convert these UI components into JSON format:
    
    Input:
    [
  {
    "type": "text_field",
    "text": "5m323e",
    "position": {
      "x": 469,
      "y": 221
    },
    "size": {
      "width": 325,
      "height": 104
    }
  }
]

    Output:
    {
      "UI": {
        "components": [
          {
            "type": "text_field",
            "text": "Enter Username",
            "position": {"x": 50, "y": 100},
            "size": {"width": 300, "height": 40}
          }
        ]
      }
    }
    
    The JSON output should be a dictionary with the components as the value.
    """

    def __init__(self, components):
        self.components = components

    def to_json(self):
        return {"UI": {"components": self.components}}

# Testing


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"  

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


In [ ]:
import torch

prompt = "Create an HTML login form with username and password fields."

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs, max_length=200)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


Create an HTML login form with username and password fields.


```html
<html>
<head>
    <title>Login Form</title>
</head>
<body>
    <form action="/submit_login" method="post">
        <label for="username">Username:</label><br>
        <input type="text" id="username" name="username"><br>
        <label for="password">Password:</label><br>
        <input type="password" id="password" name="password"><br>
        <input type="submit" value="Submit">
    </form>
</body>
</html>
```

This is a simple HTML form that will submit a POST request to the "/submit_login" endpoint when the user clicks the "Submit
